# Knowledge Assessment Agent

A clean notebook for the first agent in the new architecture.

What this notebook does:
- creates or updates learner profiles in SQLite
- stores learner preferences
- generates a short diagnostic quiz
- records quiz attempts and responses
- updates the learner baseline for the next agent


In [1]:
import json
import os
import sqlite3
import uuid
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import StructuredTool
from sentence_transformers import SentenceTransformer

load_dotenv()

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "backend" / "adaptive_tutor_v2.db"
SCHEMA_PATH = PROJECT_ROOT / "user_profile_schema.sql"
CHROMA_PATH = PROJECT_ROOT / "chroma_db"
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
EMBED_MODEL = os.getenv("EMBED_MODEL", "all-MiniLM-L6-v2")
PASS_THRESHOLD = 0.80

print('Project root:', PROJECT_ROOT)
print('DB path:', DB_PATH)
print('Schema path:', SCHEMA_PATH)


/Users/akankshacheeti/.pyenv/versions/3.10.4/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/akankshacheeti/Capstone Project 
DB path: /Users/akankshacheeti/Capstone Project /backend/adaptive_tutor_v2.db
Schema path: /Users/akankshacheeti/Capstone Project /user_profile_schema.sql


In [2]:
def init_db():
    conn = sqlite3.connect(DB_PATH)
    schema = SCHEMA_PATH.read_text(encoding='utf-8')
    conn.executescript(schema)
    conn.commit()
    conn.close()


init_db()
print('SQLite ready')


SQLite ready


In [3]:
@dataclass
class SessionState:
    learner_id: str | None = None
    email: str | None = None
    profile: dict | None = None
    preferences: dict | None = None
    current_topic: str | None = None
    current_subject_id: str | None = None
    context: str = ""
    generated_quiz: list | None = None
    quiz_responses: list | None = None
    diagnostic_score: dict | None = None


SESSION = SessionState()
llm = ChatGroq(model=GROQ_MODEL, api_key=os.getenv('GROQ_API_KEY'))
embed_model = SentenceTransformer(EMBED_MODEL)

print('LLM:', GROQ_MODEL)
print('Embedding model:', EMBED_MODEL)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10673.85it/s]


LLM: llama-3.3-70b-versatile
Embedding model: all-MiniLM-L6-v2


In [ ]:
def upsert_learner_profile(input_str: str) -> str:
    parts = [p.strip() for p in input_str.split('|')]
    email = parts[0] if len(parts) > 0 else ''
    full_name = parts[1] if len(parts) > 1 else 'Learner'
    age_group = parts[2] if len(parts) > 2 else None
    role = parts[3] if len(parts) > 3 else None
    preferred_language = parts[4] if len(parts) > 4 else 'English'

    if not email:
        return json.dumps({'error': 'email is required'})

    conn = sqlite3.connect(DB_PATH)
    row = conn.execute(
        'SELECT learner_id, full_name, age_group, role, preferred_language FROM learners WHERE email = ?',
        (email,)
    ).fetchone()

    if row:
        learner_id = row[0]
        conn.execute(
            '''UPDATE learners
               SET full_name = ?, age_group = ?, role = ?, preferred_language = ?, updated_at = datetime('now')
               WHERE learner_id = ?''',
            (full_name, age_group, role, preferred_language, learner_id)
        )
    else:
        learner_id = str(uuid.uuid4())
        conn.execute(
            '''INSERT INTO learners (learner_id, email, full_name, age_group, role, preferred_language)
               VALUES (?, ?, ?, ?, ?, ?)''',
            (learner_id, email, full_name, age_group, role, preferred_language)
        )

    conn.commit()
    conn.close()

    SESSION.learner_id = learner_id
    SESSION.email = email
    SESSION.profile = {
        'learner_id': learner_id,
        'email': email,
        'full_name': full_name,
        'age_group': age_group,
        'role': role,
        'preferred_language': preferred_language,
    }
    return json.dumps(SESSION.profile)


def save_learner_preferences(input_str: str) -> str:
    parts = [p.strip() for p in input_str.split('|')]
    learner_id = parts[0]
    content_format = parts[1] if len(parts) > 1 else 'mixed'
    explanation_style = parts[2] if len(parts) > 2 else 'step_by_step'
    quiz_style = parts[3] if len(parts) > 3 else 'mixed'
    learning_pace = parts[4] if len(parts) > 4 else 'normal'
    session_length = parts[5] if len(parts) > 5 else '30_min'
    feedback_style = parts[6] if len(parts) > 6 else 'immediate'
    accessibility_notes = parts[7] if len(parts) > 7 else None

    conn = sqlite3.connect(DB_PATH)
    existing = conn.execute(
        'SELECT preference_id FROM learner_preferences WHERE learner_id = ?',
        (learner_id,)
    ).fetchone()

    if existing:
        conn.execute(
            '''UPDATE learner_preferences
               SET content_format = ?, explanation_style = ?, quiz_style = ?, learning_pace = ?, session_length = ?, feedback_style = ?, accessibility_notes = ?, updated_at = datetime('now')
               WHERE learner_id = ?''',
            (content_format, explanation_style, quiz_style, learning_pace, session_length, feedback_style, accessibility_notes, learner_id)
        )
    else:
        conn.execute(
            '''INSERT INTO learner_preferences
               (preference_id, learner_id, content_format, explanation_style, quiz_style, learning_pace, session_length, feedback_style, accessibility_notes)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)''',
            (str(uuid.uuid4()), learner_id, content_format, explanation_style, quiz_style, learning_pace, session_length, feedback_style, accessibility_notes)
        )

    conn.commit()
    conn.close()

    SESSION.preferences = {
        'content_format': content_format,
        'explanation_style': explanation_style,
        'quiz_style': quiz_style,
        'learning_pace': learning_pace,
        'session_length': session_length,
        'feedback_style': feedback_style,
        'accessibility_notes': accessibility_notes,
    }
    return json.dumps(SESSION.preferences)

In [10]:
def fetch_learner_dashboard(email: str) -> str:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    learner = conn.execute('SELECT * FROM learners WHERE email = ?', (email.strip(),)).fetchone()
    if not learner:
        conn.close()
        return json.dumps({'error': 'learner not found'})

    prefs = conn.execute('SELECT * FROM learner_preferences WHERE learner_id = ?', (learner['learner_id'],)).fetchone()
    subject = conn.execute(
        '''SELECT * FROM learner_subject_profiles
           WHERE learner_id = ? AND is_active = 1
           ORDER BY updated_at DESC LIMIT 1''',
        (learner['learner_id'],)
    ).fetchone()
    conn.close()

    return json.dumps({
        'learner': dict(learner),
        'preferences': dict(prefs) if prefs else None,
        'active_subject_profile': dict(subject) if subject else None,
    }, default=str)


def retrieve_context(topic: str) -> str:
    topic = topic.strip()
    if not CHROMA_PATH.exists():
        return f'No chroma database found at {CHROMA_PATH}'

    import chromadb
    client = chromadb.PersistentClient(path=str(CHROMA_PATH))
    collection = client.get_or_create_collection('knowledge_base')
    if collection.count() == 0:
        return f'No material found for {topic}'

    emb = embed_model.encode([topic])[0].tolist()
    res = collection.query(query_embeddings=[emb], n_results=min(5, collection.count()))
    docs = res.get('documents', [[]])[0]
    ctx = '\n\n'.join(docs)
    SESSION.context = ctx
    return ctx


def upsert_subject(subject_name: str, description: str | None = None) -> str:
    subject_name = subject_name.strip()
    if not subject_name:
        return json.dumps({'error': 'subject_name is required'})

    conn = sqlite3.connect(DB_PATH)
    row = conn.execute(
        'SELECT subject_id, subject_name, description FROM subjects WHERE subject_name = ?',
        (subject_name,)
    ).fetchone()

    if row:
        subject_id = row[0]
        if description is not None:
            conn.execute(
                'UPDATE subjects SET description = ? WHERE subject_id = ?',
                (description, subject_id)
            )
    else:
        subject_id = str(uuid.uuid4())
        conn.execute(
            'INSERT INTO subjects (subject_id, subject_name, description) VALUES (?, ?, ?)',
            (subject_id, subject_name, description)
        )

    conn.commit()
    conn.close()
    SESSION.current_subject_id = subject_id
    return json.dumps({'subject_id': subject_id, 'subject_name': subject_name, 'description': description})


In [6]:
def generate_diagnostic_questions(input_str: str) -> str:
    parts = [p.strip() for p in input_str.split('|')]
    topic = parts[0] if len(parts) > 0 else 'General Knowledge'
    level = parts[1] if len(parts) > 1 else 'beginner'
    n = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 5

    ctx = SESSION.context or retrieve_context(topic)
    prompt = f'''You are a diagnostic quiz writer.
Create exactly {n} MCQ questions for the topic: {topic}
Learner level: {level}
Use only the context below. Return valid JSON only.
Each item should contain: id, question, options, correct_answer, explanation.

Context:
{ctx[:7000]}
'''

    resp = llm.invoke(prompt)
    raw = getattr(resp, 'content', str(resp)).strip()
    start, end = raw.find('['), raw.rfind(']')
    if start != -1 and end != -1:
        raw = raw[start:end+1]
    try:
        quiz = json.loads(raw)
        SESSION.generated_quiz = quiz
        return json.dumps(quiz)
    except Exception as exc:
        return json.dumps({'error': str(exc), 'raw': raw[:500]})


def run_diagnostic_quiz(_: str = '') -> str:
    quiz = SESSION.generated_quiz or []
    if not quiz:
        return 'ERROR: generate_diagnostic_questions first.'

    responses = []
    for q in quiz:
        print(f"\nQ{q.get('id')}: {q.get('question')}")
        for key, value in q.get('options', {}).items():
            print(f'  {key}) {value}')
        ans = input('Your answer (A/B/C/D): ').strip().upper()
        is_correct = ans == q.get('correct_answer')
        responses.append({
            'question_id': q.get('id'),
            'question': q.get('question'),
            'user_answer': ans,
            'correct_answer': q.get('correct_answer'),
            'is_correct': is_correct,
            'explanation': q.get('explanation', ''),
        })

    SESSION.quiz_responses = responses
    return f'Quiz complete: {len(responses)} responses recorded.'


def score_and_save_attempt(input_str: str) -> str:
    parts = [p.strip() for p in input_str.split('|')]
    topic = parts[0] if len(parts) > 0 else 'General Knowledge'
    session_type = parts[1] if len(parts) > 1 else 'diagnostic'

    responses = SESSION.quiz_responses or []
    if not responses:
        return json.dumps({'error': 'no quiz responses in session'})

    total = len(responses)
    correct = sum(1 for r in responses if r.get('is_correct'))
    score_pct = round(correct / total if total else 0.0, 3)
    passed = score_pct >= PASS_THRESHOLD

    learner_id = SESSION.learner_id
    if learner_id:
        attempt_id = str(uuid.uuid4())
        conn = sqlite3.connect(DB_PATH)
        conn.execute(
            '''INSERT INTO quiz_attempts
               (attempt_id, learner_id, subject_id, path_id, step_id, quiz_type, difficulty_level, score, total_questions, correct_answers, completion_status, mastery_delta, started_at, completed_at)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, datetime('now'), datetime('now'))''',
            (attempt_id, learner_id, SESSION.current_subject_id or '', None, None, session_type, level, score_pct, total, correct, 'completed', 0.0)
        )
        for item in responses:
            conn.execute(
                '''INSERT INTO quiz_responses
                   (response_id, attempt_id, question_id, question_text, selected_answer, correct_answer, is_correct, time_taken_seconds)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
                (str(uuid.uuid4()), attempt_id, str(item.get('question_id', '')), item.get('question', ''), item.get('user_answer', ''), item.get('correct_answer', ''), int(item.get('is_correct', False)), None)
            )
        conn.commit()
        conn.close()

    result = {
        'topic': topic,
        'total': total,
        'correct': correct,
        'score_pct': score_pct,
        'passed': passed,
    }
    SESSION.diagnostic_score = result
    return json.dumps(result)


In [ ]:
tool_upsert_profile = StructuredTool.from_function(
    func=upsert_learner_profile,
    name='upsert_learner_profile',
    description='Create or update a learner profile using input format email|full_name|age_group|role|preferred_language.'
)

tool_save_preferences = StructuredTool.from_function(
    func=save_learner_preferences,
    name='save_learner_preferences',
    description='Save learner preferences using input format learner_id|content_format|explanation_style|quiz_style|learning_pace|session_length.'
)
tool_fetch_dashboard = StructuredTool.from_function(
    func=fetch_learner_dashboard,
    name='fetch_learner_dashboard',
    description='Fetch learner profile, preferences, and current subject state by email.'
)

tool_retrieve_context = StructuredTool.from_function(
    func=retrieve_context,
    name='retrieve_context',
    description='Retrieve study context for a topic from ChromaDB.'
)

tool_generate_diag = StructuredTool.from_function(
    func=generate_diagnostic_questions,
    name='generate_diagnostic_questions',
    description='Generate diagnostic MCQs using input format topic|level|n_questions.'
)

tool_run_diag = StructuredTool.from_function(
    func=run_diagnostic_quiz,
    name='run_diagnostic_quiz',
    description='Run the diagnostic quiz interactively from SESSION.'
)

tool_score_attempt = StructuredTool.from_function(
    func=score_and_save_attempt,
    name='score_and_save_attempt',
    description='Score the diagnostic attempt and save it to SQLite using input format topic|diagnostic.'
)

print('Tools ready')


Tools ready


In [8]:
from langchain.agents import create_agent

knowledge_assessment_agent = create_agent(
    model=llm,
    tools=[
        tool_retrieve_context,
        tool_generate_diag,
        tool_run_diag,
        tool_score_attempt,
    ],
)

def run_knowledge_assessment(email: str, full_name: str, topic: str, level: str = 'beginner'):
    SESSION.current_topic = topic

    print('1) Upserting learner profile')
    profile = json.loads(upsert_learner_profile(f'{email}|{full_name}|None|student|English'))
    print(profile)

    print('2) Saving preferences')
    prefs = json.loads(save_learner_preferences(f"{profile['learner_id']}|mixed|step-by-step|mixed|steady|20 min"))
    print(prefs)

    print('3) Running assessment agent')
    result = knowledge_assessment_agent.invoke({
        'messages': [
            {
                'role': 'user',
                'content': (
                    f'You are a Knowledge Assessment Agent. Assess {full_name} on {topic} at {level} level. '
                    f'First retrieve context, then generate 5 questions, then run the quiz, then score and save the attempt.'
                )
            }
        ]
    })
    return result


## Quick test

Use the cells below to run the first agent manually.


In [9]:
# Example:
run_knowledge_assessment('akanksha@example.com', 'Akanksha', 'Financial Statements', 'beginner')

print('Notebook loaded. Run run_knowledge_assessment(...) when ready.')


1) Upserting learner profile
{'learner_id': 'learner-0001', 'email': 'akanksha@example.com', 'full_name': 'Akanksha', 'age_group': 'None', 'role': 'student', 'preferred_language': 'English'}
2) Saving preferences


OperationalError: no such column: pace